# 01 - Generate AnnData from 10x + CRISPR Screen Data

Build a combined AnnData object from 10x Genomics RNA and CRISPR guide
count matrices for a combinatorial E3 ligase knockout experiment.

## Imports and configuration

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns

from config import (
    ALL_GUIDE_COLUMNS,
    ANNDATA_RAW,
    CRISPR_RAW_COLUMNS,
    PROJECT_DIR,
)

os.chdir(PROJECT_DIR)

## Load sample sheet

In [ ]:
sample_sheet = pd.read_csv("./TextFiles/SampleSheet.csv")
sample_sheet

## Helper function: load one sample

For each sample, read the 10x RNA matrix and the CRISPR guide UMI counts,
intersect barcodes, clean up the guide matrix, and attach QC metrics.

In [ ]:
def load_sample(sample_name: str, sample_sheet: pd.DataFrame) -> sc.AnnData:
    """Load and merge RNA + CRISPR data for a single sample."""
    rows = sample_sheet[sample_sheet["Sample"] == sample_name]
    rna_lib = rows.loc[rows["DataType"] == "rna", "Library"].values[0]
    crispr_lib = rows.loc[rows["DataType"] == "crispr", "Library"].values[0]

    # --- RNA ---
    adata = sc.read_10x_h5(
        f"./count_outputs/{rna_lib}_SAM24459499/filtered_feature_bc_matrix.h5"
    )
    adata.var_names_make_unique()

    # --- CRISPR guide counts ---
    crispr = pd.read_csv(
        f"./count_outputs/{crispr_lib}_SAM24459499/{crispr_lib}_SAM24459499.crispr.csv",
        index_col=0,
    ).T
    crispr.index = crispr.index + "-1"

    # Select and rename guide columns
    crispr = crispr[CRISPR_RAW_COLUMNS].copy()
    crispr.columns = ALL_GUIDE_COLUMNS

    # Intersect barcodes
    shared_barcodes = adata.obs_names.intersection(crispr.index)
    adata = adata[shared_barcodes, :].copy()
    crispr = crispr.loc[shared_barcodes]

    # Set UMI count of 1 to 0 (noise filter)
    crispr[crispr == 1] = 0

    # Attach guide info to obs
    for col in ALL_GUIDE_COLUMNS:
        adata.obs[col] = crispr[col].values

    # QC columns
    adata.obs["n_umis"] = np.array(adata.X.sum(axis=1)).flatten()
    adata.obs["n_genes"] = np.array((adata.X > 0).sum(axis=1)).flatten()
    adata.obs["log10_n_umis"] = np.log10(adata.obs["n_umis"])
    adata.obs["log10_n_genes"] = np.log10(adata.obs["n_genes"])
    adata.obs["sample_name"] = sample_name

    print(f"  {sample_name}: {adata.n_obs} cells, {adata.n_vars} genes")
    return adata

## Load all samples

In [ ]:
samples = sample_sheet["Sample"].unique()
print(f"Loading {len(samples)} samples...")

ads = [load_sample(s, sample_sheet) for s in samples]

print(f"\nTotal cells before concatenation: {sum(a.n_obs for a in ads)}")

## Concatenate and filter

In [ ]:
adata = sc.AnnData.concatenate(*ads, join="inner", batch_key="sample_name")
print(f"After concatenation: {adata.n_obs} cells, {adata.n_vars} genes")

# Keep only cells with at least one guide UMI
guide_umi_sum = adata.obs[ALL_GUIDE_COLUMNS].sum(axis=1)
adata = adata[guide_umi_sum > 0, :].copy()
print(f"After guide UMI filter (sum > 0): {adata.n_obs} cells")

## Visualize guide UMI distributions

Faceted histograms of per-guide UMI counts (excluding zeros), with x-axis
clipped to 0--50.

In [ ]:
# Reshape guide UMI data to long format for plotting
guide_data = adata.obs[ALL_GUIDE_COLUMNS].copy()
guide_long = guide_data.melt(var_name="guide", value_name="umi_count")
guide_long = guide_long[guide_long["umi_count"] > 0]

n_guides = len(ALL_GUIDE_COLUMNS)
ncols = 4
nrows = int(np.ceil(n_guides / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3 * nrows), sharey=False)
axes = axes.flatten()

for i, guide in enumerate(ALL_GUIDE_COLUMNS):
    ax = axes[i]
    subset = guide_long[guide_long["guide"] == guide]["umi_count"]
    ax.hist(subset, bins=50, range=(0, 50), color="steelblue", edgecolor="white", linewidth=0.3)
    ax.set_title(guide, fontsize=10)
    ax.set_xlim(0, 50)
    ax.set_xlabel("UMI count")
    ax.set_ylabel("Cells")

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Guide UMI distributions (zeros excluded)", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## Save AnnData

In [ ]:
adata.write(ANNDATA_RAW)
print(f"Saved {adata.n_obs} cells x {adata.n_vars} genes to:\n  {ANNDATA_RAW}")